In [1]:
%pip install -q cecil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [2]:
import getpass
import os

if not os.environ.get("CECIL_API_KEY"):
    os.environ["CECIL_API_KEY"] = getpass.getpass("Cecil API key: ")

Cecil API key: ··········


In [3]:
from cecil import Client

client = Client()

In [4]:
import cecil

client = cecil.Client()

aoi = client.create_aoi(
    external_ref="Chaco Region",
    geometry={
        "type": "Polygon",
        "coordinates": [[
            [-60.3095, -20.9585],
            [-60.3095, -20.99875],
            [-60.2665, -20.99875],
            [-60.2665, -20.9585],
            [-60.3095, -20.9585],
        ]]
    }
)

print(aoi)

id='14bd0156-c196-4673-a342-be0b69c7533d' external_ref='Chaco Region' geometry={'type': 'Polygon', 'coordinates': [[[-60.3095, -20.9585], [-60.3095, -20.99875], [-60.2665, -20.99875], [-60.2665, -20.9585], [-60.3095, -20.9585]]]} hectares=1992.5956959026337 geometry_type='Polygon' location_count=None created_at=datetime.datetime(2026, 10, 8, 6, 17, 20, 326000, tzinfo=TzInfo(0)) created_by='1d3e8584-9e94-454e-b833-db4cb85fe8b0' archived_at=None archived_by=None


In [5]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
print(list(uploaded.keys()))

Saving Paglam_CR_Demarcation2025.kmz to Paglam_CR_Demarcation2025.kmz
Uploaded files:
['Paglam_CR_Demarcation2025.kmz']


In [6]:
import zipfile
import os

kmz_file = "Paglam_CR_Demarcation2025.kmz"
extract_dir = "paglam_kmz"

with zipfile.ZipFile(kmz_file, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extracted files:")
for root, dirs, files_in_dir in os.walk(extract_dir):
    for file in files_in_dir:
        print(os.path.join(root, file))

Extracted files:
paglam_kmz/doc.kml


In [7]:
import geopandas as gpd

kml_path = "paglam_kmz/doc.kml"

gdf = gpd.read_file(kml_path, driver="KML")

print("Number of features:", len(gdf))
print("CRS:", gdf.crs)
print("Geometry types:")
print(gdf.geometry.geom_type.value_counts())

display(gdf)

Number of features: 1
CRS: EPSG:4326
Geometry types:
Polygon    1
Name: count, dtype: int64


,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
0,None,Paglam_CR_Demarcation2025_final,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"POLYGON Z ((95.49388 27.83197 0, 95.49219 27.8..."


In [8]:
print("Original CRS:", gdf.crs)

# Ensure WGS84 / latitude-longitude
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
else:
    gdf = gdf.to_crs("EPSG:4326")

# Remove empty geometries
gdf = gdf[~gdf.geometry.is_empty].copy()

# Fix invalid geometries if necessary
gdf["geometry"] = gdf.geometry.buffer(0)

print("Final CRS:", gdf.crs)
print("Features:", len(gdf))
print("Valid geometries:", gdf.geometry.is_valid.all())

display(gdf)

Original CRS: EPSG:4326
Final CRS: EPSG:4326
Features: 1
Valid geometries: True


,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,geometry
0,None,Paglam_CR_Demarcation2025_final,None,NaT,NaT,NaT,None,1,0,-1,NaN,None,"POLYGON Z ((95.49388 27.83197 0, 95.49394 27.8..."


In [18]:
from shapely.geometry import mapping, Polygon, MultiPolygon

def remove_z(geom):
    """Convert 3D geometry to 2D geometry."""

    if geom.geom_type == "Polygon":
        exterior = [(x, y) for x, y, *rest in geom.exterior.coords]
        interiors = [
            [(x, y) for x, y, *rest in ring.coords]
            for ring in geom.interiors
        ]
        return Polygon(exterior, interiors)

    elif geom.geom_type == "MultiPolygon":
        polygons = []

        for poly in geom.geoms:
            exterior = [(x, y) for x, y, *rest in poly.exterior.coords]
            interiors = [
                [(x, y) for x, y, *rest in ring.coords]
                for ring in poly.interiors
            ]
            polygons.append(Polygon(exterior, interiors))

        return MultiPolygon(polygons)

    else:
        raise ValueError(f"Unsupported geometry type: {geom.geom_type}")


paglam_geometry_2d = remove_z(paglam_geometry)

print("Original geometry:", paglam_geometry.geom_type)
print("2D geometry:", paglam_geometry_2d.geom_type)
print("Has Z:", paglam_geometry_2d.has_z)
print("Valid:", paglam_geometry_2d.is_valid)

paglam_geojson = mapping(paglam_geometry_2d)

print("GeoJSON type:", paglam_geojson["type"])

Original geometry: Polygon
2D geometry: Polygon
Has Z: False
Valid: True
GeoJSON type: Polygon


In [19]:
# Drop the Z coordinate (3D to 2D conversion) from the GeoJSON polygon
def strip_z_coordinate(geometry_dict):
    if geometry_dict["type"] == "Polygon":
        new_coords = []
        for ring in geometry_dict["coordinates"]:
            new_coords.append([coord[:2] for coord in ring])
        geometry_dict["coordinates"] = new_coords
    return geometry_dict

paglam_geojson_2d = strip_z_coordinate(paglam_geojson)

# Create the Cecil AOI using the 2D Paglam geometry
aoi_response = client.create_aoi(
    external_ref="Paglam_CR_Demarcation2025",
    geometry=paglam_geojson_2d
)

# Assign AOI_ID globally so the main workflow can detect it
AOI_ID = aoi_response.id
print("✓ Created Cecil AOI successfully!")
print("AOI_ID assigned:", AOI_ID)

✓ Created Cecil AOI successfully!
AOI_ID assigned: 4db972f3-1895-415e-8b2a-30d2163649da


In [20]:
print("Geometry type:", paglam_geojson["type"])

if paglam_geojson["type"] == "Polygon":
    print("Number of rings:", len(paglam_geojson["coordinates"]))
    print("Outer boundary points:", len(paglam_geojson["coordinates"][0]))

elif paglam_geojson["type"] == "MultiPolygon":
    print("Number of polygons:", len(paglam_geojson["coordinates"]))

Geometry type: Polygon
Number of rings: 1
Outer boundary points: 58


In [21]:
area_gdf = gpd.GeoDataFrame(
    {"name": ["Paglam_CR_Demarcation2025"]},
    geometry=[paglam_geometry],
    crs="EPSG:4326"
)

area_m2 = area_gdf.to_crs("EPSG:6933").geometry.area.iloc[0]
area_km2 = area_m2 / 1_000_000

print(f"Paglam boundary area: {area_km2:.3f} km²")

Paglam boundary area: 2.518 km²


In [22]:
WWF_DATASET_ID = "2662e5cd-662e-4f65-a3de-5d217beb7111"

wwf = client.get_dataset(WWF_DATASET_ID)

print(wwf)

id='2662e5cd-662e-4f65-a3de-5d217beb7111' name='Biodiversity Risk Filter' type='Vector' crs='EPSG:4326' description=['This dataset contains biodiversity risk indicators for terrestrial, freshwater, and marine areas, with global coverage at sub-basin, marine ecoregion, or open ocean fishing area scale (polygons). Fifty-five indicators are aggregated into fifteen risk categories and three risk types: physical risk, regulatory deficiency risk, and reputational risk. This dataset is part of the WWF Risk Filter Suite: translating complex environmental data and nature-related risk concepts into actionable insights for business.', 'Assessment units follow HydroBASINS Level 7 for terrestrial and freshwater areas, Marine Ecoregions of the World for coastal marine areas, and sub-areas of the FAO Major Fishing Areas for open ocean areas, with terrestrial units that cross national borders sub-divided along national boundaries. World Bank boundaries are used for island nations not covered by HydroB

In [23]:
aoi = client.create_aoi(
    external_ref="Paglam_CR_Demarcation2025",
    geometry=paglam_geojson
)

print(aoi)

id='4db972f3-1895-415e-8b2a-30d2163649da' external_ref='Paglam_CR_Demarcation2025' geometry={'type': 'Polygon', 'coordinates': [[[95.493879616, 27.831972249], [95.493942866, 27.8318117], [95.494149962, 27.831320951], [95.494397314, 27.830689334], [95.49474032, 27.829885893], [95.494858021, 27.829095513], [95.494862058, 27.828502144], [95.494925032, 27.827812572], [95.495370266, 27.826652028], [95.495354931, 27.826148178], [95.494845887, 27.825455526], [95.494467139, 27.824965837], [95.494383257, 27.824443454], [95.494671774, 27.824056325], [95.495231996, 27.823638401], [95.496019361, 27.823498177], [95.496522069, 27.82350171], [95.497001659, 27.823634787], [95.497881597, 27.823407162], [95.498665315, 27.823360573], [95.499846982, 27.823214384], [95.500967566, 27.823123991], [95.502142049, 27.822999524], [95.502757796, 27.823009222], [95.503226923, 27.822862721], [95.503680972, 27.821307253], [95.502653138, 27.81993633], [95.498481612, 27.817119931], [95.496878414, 27.816568397], [95.49

In [25]:
FREE_DATASETS = {
    "Hansen_Global_Forest_Change":
        "9659ec1d-7091-4f8b-9db5-e9fe07d2f508",

    "JRC_Global_Forest_Cover":
        "c17fcd5f-02bc-42e4-b069-779b17d2655a",

    "WRI_SBTN_Natural_Lands":
        "bf249342-557a-4028-922c-c67fc4ad6a64",
}

In [27]:
# List all datasets available to your Cecil account
datasets = client.list_datasets()

print("FREE / OPEN DATASETS")
print("=" * 80)

free_datasets = []

for d in datasets:
    licence = getattr(d, "licence", None)

    if licence is not None:
        licence_type = str(getattr(licence, "type", "")).lower()

        if licence_type == "open":
            free_datasets.append(d)

for i, d in enumerate(free_datasets, 1):
    print(f"{i}. {d.name}")
    print(f"   ID   : {d.id}")
    print(f"   Type : {d.type}")
    print()

print("Total free/open datasets:", len(free_datasets))

FREE / OPEN DATASETS
1. Global Aboveground Biomass 100 m
   ID   : 202b37bb-23bf-47f8-bc71-558292d69c31
   Type : Raster

2. Comprehensive Global Administrative Zones - ADM0 Countries
   ID   : 2d986cf5-71be-4b42-aeba-ccd27ac5f592
   Type : Vector

3. Comprehensive Global Administrative Zones - ADM1 Districts
   ID   : 54c93139-9efe-491c-82c4-dd685a8cd209
   Type : Vector

4. Comprehensive Global Administrative Zones - ADM2 Municipalities
   ID   : 078dea69-469f-4257-a72e-43cb4827a13a
   Type : Vector

5. Land Cover 9-Class
   ID   : a4bb9aea-b6df-4d19-9083-38357f8fa76c
   Type : Raster

6. Global Forest Cover
   ID   : 9cfbe1d5-040d-4c13-8151-87ebdce8e0c2
   Type : Raster

7. Global Forest Type
   ID   : c17fcd5f-02bc-42e4-b069-779b17d2655a
   Type : Raster

8. Hansen Global Forest Change
   ID   : 9659ec1d-7091-4f8b-9db5-e9fe07d2f508
   Type : Raster

9. Cropland Data Layer 10 m
   ID   : 84f5bd14-83fc-4b45-8a65-ebe31eea4da1
   Type : Raster

10. Cropland Data Layer 30 m
   ID   : 86

In [28]:
BIOMASS_DATASET_ID = "202b37bb-23bf-47f8-bc71-558292d69c31"

biomass_dataset = client.get_dataset(BIOMASS_DATASET_ID)

print("Dataset name:", biomass_dataset.name)
print("Dataset ID:", biomass_dataset.id)
print("Dataset type:", biomass_dataset.type)
print("\nDataset details:")
print(biomass_dataset)

Dataset name: Global Aboveground Biomass 100 m
Dataset ID: 202b37bb-23bf-47f8-bc71-558292d69c31
Dataset type: Raster

Dataset details:
id='202b37bb-23bf-47f8-bc71-558292d69c31' name='Global Aboveground Biomass 100 m' type='Raster' crs='EPSG:4326' description=['This dataset quantifies aboveground biomass density (Mg/ha) at 100 m spatial resolution, with global coverage annually from 2000 onwards. Data includes pixel-level residual uncertainty (± standard error).'] usage_notes=['New time points are delivered annually. Spatial coverage is limited to vegetated land surfaces between 81.8°N and 81.8°S. Estimates for years before 2010 and for regions with sparse reference samples rely more heavily on temporal extrapolation, so are better suited to identifying long-term trends than to precise annual stock assessment.'] categories=['Plant biomass'] licence=Licence(type='Open') version=Version(date='2026-08-25', number='1.2.0') spatial_coverage=SpatialCoverage(nominal='Global') spatial_resolutio

In [29]:
BIOMASS_DATASET_ID = "202b37bb-23bf-47f8-bc71-558292d69c31"

biomass_subscription = client.create_subscription(
    aoi_id=AOI_ID,
    dataset_id=BIOMASS_DATASET_ID,
    external_ref="Paglam_CR_Demarcation2025_Global_Biomass_100m"
)

print(biomass_subscription)

id='3502b7e7-8012-4c06-9e2f-5050315dac63' aoi_id='4db972f3-1895-415e-8b2a-30d2163649da' dataset_id='202b37bb-23bf-47f8-bc71-558292d69c31' dataset_publication='1.2.0' dataset_current_publication='1.2.0' status='pending' status_message=None external_ref='Paglam_CR_Demarcation2025_Global_Biomass_100m' created_at=datetime.datetime(2026, 10, 8, 6, 29, 8, 922000, tzinfo=TzInfo(0)) created_by='1d3e8584-9e94-454e-b833-db4cb85fe8b0' archived_at=None archived_by=None


In [30]:
BIOMASS_SUBSCRIPTION_ID = biomass_subscription.id

print("Biomass subscription ID:")
print(BIOMASS_SUBSCRIPTION_ID)

Biomass subscription ID:
3502b7e7-8012-4c06-9e2f-5050315dac63


In [31]:
status = client.get_subscription(BIOMASS_SUBSCRIPTION_ID)

print("Status:", status.status)
print("Message:", status.status_message)

Status: processing
Message: None


In [32]:
import time

while True:
    status = client.get_subscription(BIOMASS_SUBSCRIPTION_ID)

    print(
        "Status:",
        status.status,
        "|",
        status.status_message
    )

    if status.status == "completed":
        print("\nBiomass extraction completed!")
        break

    if status.status == "failed":
        raise RuntimeError(
            f"Biomass extraction failed: {status.status_message}"
        )

    time.sleep(30)

Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: processing | None
Status: completed | None

Biomass extraction completed!


In [34]:
# Since this is a Raster dataset, use load_xarray instead of load_dataframe
biomass_ds = client.load_xarray(
    subscription_id=BIOMASS_SUBSCRIPTION_ID
)

print("Data type:", type(biomass_ds))
print("Dimensions:", biomass_ds.dims)

print("\nVariables:")
print(list(biomass_ds.data_vars))

display(biomass_ds)

Data type: <class 'xarray.core.dataset.Dataset'>
Dimensions: FrozenMappingWarningOnValuesAccess({'y': 21, 'x': 23, 'time': 26})

Variables:
['aboveground_biomass_density', 'aboveground_biomass_density_uncertainty']


<xarray.Dataset> Size: 101kB
Dimensions:                                  (y: 21, x: 23, time: 26)
Coordinates:
  * y                                        (y) float64 168B 27.83 ... 27.81
  * x                                        (x) float64 184B 95.48 ... 95.5
  * time                                     (time) datetime64[ns] 208B 2000-...
    spatial_ref                              int64 8B 0
Data variables:
    aboveground_biomass_density              (time, y, x) float32 50kB dask.array<chunksize=(1, 21, 23), meta=np.ndarray>
    aboveground_biomass_density_uncertainty  (time, y, x) float32 50kB dask.array<chunksize=(1, 21, 23), meta=np.ndarray>
Attributes:
    provider_name:                CTrees
    dataset_name:                 Global Aboveground Biomass 100 m
    dataset_id:                   202b37bb-23bf-47f8-bc71-558292d69c31
    aoi_id:                       4db972f3-1895-415e-8b2a-30d2163649da
    subscription_id:              3502b7e7-8012-4c06-9e2f-5050315dac63
    dataset_publication:          1.2.0
    dataset_current_publication:  1.2.0

In [37]:
import numpy as np
import pandas as pd

# Using biomass_ds (xarray Dataset) instead of biomass_df
biomass = biomass_ds["aboveground_biomass_density"]

# Calculate statistics across all pixels inside the returned AOI grid
annual_stats = pd.DataFrame({
    "year": biomass_ds.time.dt.year.values,
    "mean_biomass_Mg_ha": biomass.mean(dim=["y", "x"]).compute().values,
    "median_biomass_Mg_ha": biomass.median(dim=["y", "x"]).compute().values,
    "min_biomass_Mg_ha": biomass.min(dim=["y", "x"]).compute().values,
    "max_biomass_Mg_ha": biomass.max(dim=["y", "x"]).compute().values,
    "std_biomass_Mg_ha": biomass.std(dim=["y", "x"]).compute().values,
    "valid_pixel_count": biomass.notnull().sum(dim=["y", "x"]).compute().values
})

print("Paglam Annual Aboveground Biomass")
display(annual_stats)

Paglam Annual Aboveground Biomass


,year,mean_biomass_Mg_ha,median_biomass_Mg_ha,min_biomass_Mg_ha,max_biomass_Mg_ha,std_biomass_Mg_ha,valid_pixel_count
0,2000,23.444521,20.400000,4.5,143.900009,15.195508,292
1,2001,23.470205,20.549999,2.8,131.500000,14.492593,292
2,2002,24.386303,21.900000,4.0,134.300003,14.060695,292
3,2003,26.468493,24.049999,5.1,142.800003,14.023768,292
4,2004,27.653767,26.100000,4.3,140.900009,13.955599,292
5,2005,27.886986,25.900000,6.0,136.699997,13.139094,292
6,2006,28.394863,26.549999,5.7,134.400009,13.318879,292
7,2007,29.018494,27.150002,6.1,125.200005,12.592858,292
8,2008,28.872602,27.650002,6.2,140.500000,13.290804,292
9,2009,29.739725,27.950001,7.2,128.000000,13.261206,292


In [39]:
paglam_stats = pd.DataFrame({
    "year": biomass_ds.time.dt.year.values,

    "mean_biomass_Mg_ha": biomass.mean(
        dim=["y", "x"]
    ).compute().values,

    "median_biomass_Mg_ha": biomass.median(
        dim=["y", "x"]
    ).compute().values,

    "min_biomass_Mg_ha": biomass.min(
        dim=["y", "x"]
    ).compute().values,

    "max_biomass_Mg_ha": biomass.max(
        dim=["y", "x"]
    ).compute().values,

    "std_biomass_Mg_ha": biomass.std(
        dim=["y", "x"]
    ).compute().values,

    "valid_pixel_count": biomass.notnull().sum(
        dim=["y", "x"]
    ).compute().values
})

display(paglam_stats)

,year,mean_biomass_Mg_ha,median_biomass_Mg_ha,min_biomass_Mg_ha,max_biomass_Mg_ha,std_biomass_Mg_ha,valid_pixel_count
0,2000,23.444521,20.400000,4.5,143.900009,15.195508,292
1,2001,23.470205,20.549999,2.8,131.500000,14.492593,292
2,2002,24.386303,21.900000,4.0,134.300003,14.060695,292
3,2003,26.468493,24.049999,5.1,142.800003,14.023768,292
4,2004,27.653767,26.100000,4.3,140.900009,13.955599,292
5,2005,27.886986,25.900000,6.0,136.699997,13.139094,292
6,2006,28.394863,26.549999,5.7,134.400009,13.318879,292
7,2007,29.018494,27.150002,6.1,125.200005,12.592858,292
8,2008,28.872602,27.650002,6.2,140.500000,13.290804,292
9,2009,29.739725,27.950001,7.2,128.000000,13.261206,292


In [49]:
# =============================================================================
# PAGLAM CECIL NATURE METRICS — CORRECTED STATISTICS-ONLY VERSION
# =============================================================================
#
# PURPOSE
# -------
# Extract statistically valid Cecil dataset metrics for the exact Paglam KML
# boundary.
#
# IMPORTANT METHODOLOGICAL RULE
# -----------------------------
# NO artificial 0–10 normalization is performed.
#
# We report:
#   1. Raw/statistical values
#   2. Source-provided scores where they genuinely exist
#   3. Categorical class proportions
#
# We DO NOT convert:
#   percentage -> arbitrary 0–10 score
#   Aqueduct score -> another 0–10 score
#   forest loss -> invented risk score
#
# Age & Sex dataset is explicitly excluded.
# =============================================================================


import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray


# =============================================================================
# 0. CHECK REQUIRED EXISTING OBJECTS
# =============================================================================

print("=" * 90)
print("PAGLAM CECIL NATURE METRICS — STATISTICS ONLY")
print("=" * 90)

required_objects = [
    "client",
    "AOI_ID",
    "boundary_gdf"
]

missing = [x for x in required_objects if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing required existing objects: {missing}\n"
        "Please run your Cecil authentication/AOI/boundary cells first."
    )

print(f"AOI_ID: {AOI_ID}")
print("Age & Sex dataset: EXCLUDED")
print()


# =============================================================================
# 1. CECIL DATASET IDs
# =============================================================================

DATASETS = {

    # -------------------------------------------------------------------------
    # Natural land
    # -------------------------------------------------------------------------
    "SBTN Natural Lands":
        "bf249342-557a-4028-922c-c67fc4ad6a64",

    # -------------------------------------------------------------------------
    # Land cover
    # -------------------------------------------------------------------------
    "Land Cover 9-Class":
        "a4bb9aea-b6df-4d19-9083-38357f8fa76c",

    # -------------------------------------------------------------------------
    # Forest
    # -------------------------------------------------------------------------
    "Global Forest Cover":
        "9cfbe1d5-040d-4c13-8151-87ebdce8e0c2",

    "Global Forest Type":
        "c17fcd5f-02bc-42e4-b069-779b17d2655a",

    "Hansen Global Forest Change":
        "9659ec1d-7091-4f8b-9db5-e9fe07d2f508",

    # -------------------------------------------------------------------------
    # Landscape integrity
    # -------------------------------------------------------------------------
    "Forest Landscape Integrity Index":
        "03f5ee4b-2419-4c79-bb44-5f838e43a977",

    # -------------------------------------------------------------------------
    # Tropical tree cover
    # -------------------------------------------------------------------------
    "Tropical Tree Cover 10 m":
        "72cfdb7e-5c49-45d4-be8d-02ff227a24b8",

    "Tropical Tree Cover 70 m":
        "e8fe6f99-606b-4c5a-bbe8-6ae34bda02ad",

    # -------------------------------------------------------------------------
    # Water
    # -------------------------------------------------------------------------
    "Aqueduct Water Risk - Baseline Annual":
        "ef65afb9-e593-4705-be57-3bbe1c863577",

    "Aqueduct Water Risk - Baseline Monthly":
        "6075fade-09d8-4288-83e8-80e03f229767",

    "Aqueduct Water Risk - Future Annual":
        "8197700-9f9a-448c-ad74-ad989bd78549",

    # -------------------------------------------------------------------------
    # Population
    # -------------------------------------------------------------------------
    "Population Counts 1 km":
        "0273552c-b519-4ff8-906e-f4969c3a4fc7",

    # -------------------------------------------------------------------------
    # Administrative context
    # -------------------------------------------------------------------------
    "Administrative Zones - ADM0":
        "2d986cf5-71be-4b42-aeba-ccd27ac5f592",

    "Administrative Zones - ADM1":
        "54c93139-9efe-491c-82c4-dd685a8cd209",

    "Administrative Zones - ADM2":
        "078dea69-469f-4257-a72e-43cb4827a13a",
}


# =============================================================================
# 2. CORRECT THE AQUEDUCT FUTURE DATASET ID IF NECESSARY
# =============================================================================

# The known Cecil dataset ID from the existing loaded subscription is:
DATASETS["Aqueduct Water Risk - Future Annual"] = \
    "819b7700-9f9a-448c-ad74-ad989bd78549"


# =============================================================================
# 3. NO-DATA VALUES
# =============================================================================

NODATA_VALUES = [
    -9999,
    -999,
    -99999,
    -32768,
    32767,
    65535,
    255
]


# =============================================================================
# 4. HELPER — CLEAN NUMERIC ARRAY
# =============================================================================

def clean_numeric(values, remove_255=False):

    arr = np.asarray(values, dtype=float)

    arr = arr[np.isfinite(arr)]

    for nd in NODATA_VALUES:
        if nd == 255 and not remove_255:
            continue

        arr = arr[arr != nd]

    return arr


# =============================================================================
# 5. HELPER — CONVERT DATASET TO RASTER
# =============================================================================

def prepare_raster(da):

    # If a Dataset was accidentally supplied, select first data variable
    if isinstance(da, xr.Dataset):

        variables = list(da.data_vars)

        if len(variables) == 0:
            raise ValueError("No raster variables found.")

        da = da[variables[0]]

    # Identify spatial dimensions
    x_candidates = ["x", "lon", "longitude"]
    y_candidates = ["y", "lat", "latitude"]

    x_dim = next((x for x in x_candidates if x in da.dims), None)
    y_dim = next((y for y in y_candidates if y in da.dims), None)

    if x_dim is None or y_dim is None:
        raise ValueError(
            f"Could not identify spatial dimensions. Dimensions: {da.dims}"
        )

    da = da.rio.set_spatial_dims(
        x_dim=x_dim,
        y_dim=y_dim,
        inplace=False
    )

    try:
        da = da.rio.write_crs("EPSG:4326", inplace=False)
    except Exception:
        pass

    return da


# =============================================================================
# 6. HELPER — CLIP TO EXACT PAGLAM KML
# =============================================================================

def clip_to_paglam(da):

    da = prepare_raster(da)

    clipped = da.rio.clip(
        boundary_gdf.geometry,
        boundary_gdf.crs,
        drop=True,
        all_touched=False
    )

    return clipped


# =============================================================================
# 7. HELPER — REDUCE ALL NON-SPATIAL DIMENSIONS
# =============================================================================

def remove_extra_dimensions(da):

    spatial_dims = [
        d for d in da.dims
        if d.lower() in ["x", "y", "lon", "lat", "longitude", "latitude"]
    ]

    other_dims = [
        d for d in da.dims
        if d not in spatial_dims
    ]

    # If time / other dimensions exist, keep them separately.
    return spatial_dims, other_dims


# =============================================================================
# 8. HELPER — STATISTICS
# =============================================================================

def raster_statistics(da):

    """
    Returns robust statistics from valid pixels.

    Does NOT create any score.
    """

    try:
        da = clip_to_paglam(da)

        values = da.values

        values = clean_numeric(
            values.flatten(),
            remove_255=True
        )

        if len(values) == 0:
            return None

        return {
            "valid_pixel_count": int(len(values)),
            "mean": float(np.mean(values)),
            "median": float(np.median(values)),
            "minimum": float(np.min(values)),
            "maximum": float(np.max(values)),
            "std_dev": float(np.std(values))
        }

    except Exception as e:

        print(f"      Statistics error: {e}")

        return None


# =============================================================================
# 9. SUBSCRIPTION DISCOVERY
# =============================================================================

print("=" * 90)
print("LOADING / REUSING CECIL DATASETS")
print("=" * 90)


# Get current subscriptions if available
try:
    existing_subscriptions = client.list_subscriptions()
except Exception:
    existing_subscriptions = []

existing_by_dataset = {}

try:

    if hasattr(existing_subscriptions, "to_dict"):
        sub_records = existing_subscriptions.to_dict("records")
    elif isinstance(existing_subscriptions, list):
        sub_records = existing_subscriptions
    else:
        sub_records = []

    for sub in sub_records:

        dataset_id = (
            sub.get("dataset_id")
            or sub.get("dataset")
            or sub.get("datasetId")
        )

        sub_id = (
            sub.get("id")
            or sub.get("subscription_id")
        )

        if dataset_id and sub_id:
            existing_by_dataset[dataset_id] = sub_id

except Exception:
    pass


# =============================================================================
# 10. LOAD FUNCTION
# =============================================================================

loaded_data = {}
failed_data = {}


def load_cecil_dataset(name, dataset_id):

    print(f"\n{name}")

    # -------------------------------------------------------------------------
    # Try already-existing subscription object from previous notebook cells
    # -------------------------------------------------------------------------

    subscription_id = None

    # Search common notebook variable: subscriptions
    if "subscriptions" in globals():

        try:

            subs = subscriptions

            if isinstance(subs, dict):

                candidate = (
                    subs.get(name)
                    or subs.get(dataset_id)
                )

                if isinstance(candidate, dict):
                    subscription_id = (
                        candidate.get("id")
                        or candidate.get("subscription_id")
                    )
                elif isinstance(candidate, str):
                    subscription_id = candidate

        except Exception:
            pass


    # -------------------------------------------------------------------------
    # Try existing Cecil subscription list
    # -------------------------------------------------------------------------

    if subscription_id is None:

        subscription_id = existing_by_dataset.get(dataset_id)


    # -------------------------------------------------------------------------
    # Create subscription if needed
    # -------------------------------------------------------------------------

    try:

        if subscription_id is not None:

            print(f"   Reusing subscription: {subscription_id}")

        else:

            print("   Creating subscription...")

            subscription = client.create_subscription(
                dataset_id=dataset_id,
                aoi_id=AOI_ID
            )

            if isinstance(subscription, dict):

                subscription_id = (
                    subscription.get("id")
                    or subscription.get("subscription_id")
                )

            else:

                subscription_id = getattr(
                    subscription,
                    "id",
                    None
                )

                if subscription_id is None:
                    subscription_id = getattr(
                        subscription,
                        "subscription_id",
                        None
                    )

            if subscription_id is None:
                raise RuntimeError(
                    "Cecil subscription created but ID could not be identified."
                )

            print(f"   Subscription created: {subscription_id}")


        # ---------------------------------------------------------------------
        # Try xarray first
        # ---------------------------------------------------------------------

        data = None

        if hasattr(client, "load_xarray"):

            try:
                data = client.load_xarray(
                    subscription_id=subscription_id
                )

            except Exception:
                data = None


        # ---------------------------------------------------------------------
        # Fall back to load_dataframe
        # ---------------------------------------------------------------------

        if data is None:

            data = client.load_dataframe(
                subscription_id=subscription_id
            )


        loaded_data[name] = data


        # ---------------------------------------------------------------------
        # Print what was loaded
        # ---------------------------------------------------------------------

        if isinstance(data, xr.Dataset):

            print(
                "   ✓ Raster loaded:",
                list(data.data_vars)
            )

        elif isinstance(data, xr.DataArray):

            print(
                "   ✓ Raster loaded:",
                data.name
            )

        elif isinstance(data, pd.DataFrame):

            print(
                f"   ✓ Vector loaded: "
                f"{len(data)} row(s), "
                f"{len(data.columns)} columns"
            )

        else:

            print(
                f"   ✓ Loaded object: {type(data)}"
            )


    except Exception as e:

        failed_data[name] = str(e)

        print(
            f"   ✗ Error during setup for {name}: {e}"
        )


# =============================================================================
# 11. LOAD DATASETS
# =============================================================================

for dataset_name, dataset_id in DATASETS.items():

    load_cecil_dataset(
        dataset_name,
        dataset_id
    )


print("\n")
print("=" * 90)
print("LOADING COMPLETE")
print("=" * 90)


# =============================================================================
# 12. METRIC TABLE
# =============================================================================

metric_rows = []


def add_metric(
    dataset,
    metric,
    statistic,
    value,
    unit,
    valid_pixel_count=None,
    notes=""
):

    metric_rows.append({

        "site":
            "Paglam_CR_Demarcation2025",

        "aoi_id":
            AOI_ID,

        "dataset":
            dataset,

        "metric":
            metric,

        "statistic":
            statistic,

        "value":
            value,

        "unit":
            unit,

        "valid_pixel_count":
            valid_pixel_count,

        "notes":
            notes
    })


# =============================================================================
# 13. SBTN NATURAL LANDS
# =============================================================================

print("\n" + "=" * 90)
print("CALCULATING PAGLAM METRICS")
print("=" * 90)


if "SBTN Natural Lands" in loaded_data:

    print("\nSBTN Natural Lands")

    ds = loaded_data["SBTN Natural Lands"]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )

        if "binary_class" in ds:

            try:

                da = clip_to_paglam(
                    ds["binary_class"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                # SBTN binary_class:
                # 0 = non-natural
                # 1 = natural
                #
                # We report the observed percentage only.
                natural = np.sum(values == 1)

                total = len(values)

                natural_pct = (
                    natural / total * 100
                    if total > 0
                    else np.nan
                )

                add_metric(
                    "SBTN Natural Lands",
                    "Natural Land Extent",
                    "Percentage of valid pixels classified as natural",
                    natural_pct,
                    "%",
                    total,
                    "SBTN binary_class: natural vs non-natural."
                )

                add_metric(
                    "SBTN Natural Lands",
                    "Natural Land Pixel Count",
                    "Count",
                    natural,
                    "pixels",
                    total,
                    "Count of valid pixels classified as natural."
                )

            except Exception as e:

                print(
                    f"   SBTN calculation failed: {e}"
                )


# =============================================================================
# 14. LAND COVER 9-CLASS
# =============================================================================

if "Land Cover 9-Class" in loaded_data:

    print("\nLand Cover 9-Class")

    ds = loaded_data["Land Cover 9-Class"]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )

        if "land_cover_class" in ds:

            try:

                da = clip_to_paglam(
                    ds["land_cover_class"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                unique, counts = np.unique(
                    values,
                    return_counts=True
                )

                total = counts.sum()

                for cls, count in zip(unique, counts):

                    pct = (
                        count / total * 100
                        if total > 0
                        else np.nan
                    )

                    add_metric(
                        "Land Cover 9-Class",
                        f"Land Cover Class {int(cls)}",
                        "Class proportion",
                        pct,
                        "%",
                        int(total),
                        "Categorical raster. Class codes are reported without averaging."
                    )

                print("   Class proportions:")

                for cls, count in zip(unique, counts):

                    print(
                        f"      Class {int(cls)}: "
                        f"{count / total * 100:.2f}%"
                    )

            except Exception as e:

                print(
                    f"   Land Cover calculation failed: {e}"
                )


# =============================================================================
# 15. GLOBAL FOREST COVER
# =============================================================================

if "Global Forest Cover" in loaded_data:

    print("\nGlobal Forest Cover")

    ds = loaded_data["Global Forest Cover"]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )

        if "forest_cover" in ds:

            try:

                da = clip_to_paglam(
                    ds["forest_cover"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                # Forest cover is treated as binary:
                # 0 = non-forest
                # 1 = forest
                forest_pixels = np.sum(
                    values == 1
                )

                total = len(values)

                forest_pct = (
                    forest_pixels / total * 100
                    if total > 0
                    else np.nan
                )

                add_metric(
                    "Global Forest Cover",
                    "Forest Cover",
                    "Percentage of valid pixels classified as forest",
                    forest_pct,
                    "%",
                    total,
                    "Binary forest/non-forest classification. NoData excluded."
                )

                add_metric(
                    "Global Forest Cover",
                    "Forest Pixel Count",
                    "Count",
                    forest_pixels,
                    "pixels",
                    total,
                    "Count of valid forest pixels."
                )

                print(
                    f"   Forest cover: {forest_pct:.2f}%"
                )

            except Exception as e:

                print(
                    f"   Forest Cover calculation failed: {e}"
                )


# =============================================================================
# 16. GLOBAL FOREST TYPE
# =============================================================================

if "Global Forest Type" in loaded_data:

    print("\nGlobal Forest Type")

    ds = loaded_data["Global Forest Type"]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )

        if "forest_type" in ds:

            try:

                da = clip_to_paglam(
                    ds["forest_type"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                unique, counts = np.unique(
                    values,
                    return_counts=True
                )

                total = counts.sum()

                for cls, count in zip(unique, counts):

                    pct = (
                        count / total * 100
                        if total > 0
                        else np.nan
                    )

                    add_metric(
                        "Global Forest Type",
                        f"Forest Type Class {int(cls)}",
                        "Class proportion",
                        pct,
                        "%",
                        int(total),
                        "Categorical forest-type raster. Class proportions only; no class-code averaging."
                    )

                print("   Forest type classes:")

                for cls, count in zip(unique, counts):

                    print(
                        f"      Class {int(cls)}: "
                        f"{count / total * 100:.2f}%"
                    )

            except Exception as e:

                print(
                    f"   Forest Type calculation failed: {e}"
                )


# =============================================================================
# 17. HANSEN GLOBAL FOREST CHANGE
# =============================================================================

if "Hansen Global Forest Change" in loaded_data:

    print("\nHansen Global Forest Change")

    ds = loaded_data["Hansen Global Forest Change"]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )


        # ---------------------------------------------------------------------
        # TREE COVER
        # ---------------------------------------------------------------------

        if "tree_cover" in ds:

            try:

                da = clip_to_paglam(
                    ds["tree_cover"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                if len(values) > 0:

                    add_metric(
                        "Hansen Global Forest Change",
                        "Tree Cover 2000",
                        "Mean",
                        np.mean(values),
                        "%",
                        len(values),
                        "Mean Hansen tree-cover value for valid pixels."
                    )

                    add_metric(
                        "Hansen Global Forest Change",
                        "Tree Cover 2000",
                        "Median",
                        np.median(values),
                        "%",
                        len(values),
                        "Median Hansen tree-cover value for valid pixels."
                    )

                    add_metric(
                        "Hansen Global Forest Change",
                        "Tree Cover 2000",
                        "Minimum",
                        np.min(values),
                        "%",
                        len(values),
                        "Minimum valid tree-cover value."
                    )

                    add_metric(
                        "Hansen Global Forest Change",
                        "Tree Cover 2000",
                        "Maximum",
                        np.max(values),
                        "%",
                        len(values),
                        "Maximum valid tree-cover value."
                    )

                    add_metric(
                        "Hansen Global Forest Change",
                        "Tree Cover 2000",
                        "Standard Deviation",
                        np.std(values),
                        "%",
                        len(values),
                        "Standard deviation of valid tree-cover values."
                    )

                    print(
                        f"   Tree cover mean: "
                        f"{np.mean(values):.4f}%"
                    )

            except Exception as e:

                print(
                    f"   Hansen tree_cover calculation failed: {e}"
                )


        # ---------------------------------------------------------------------
        # FOREST LOSS YEAR
        # ---------------------------------------------------------------------

        if "loss_year" in ds:

            try:

                da = clip_to_paglam(
                    ds["loss_year"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                if len(values) > 0:

                    loss_pixels = np.sum(
                        values > 0
                    )

                    total = len(values)

                    loss_pct = (
                        loss_pixels / total * 100
                        if total > 0
                        else np.nan
                    )

                    add_metric(
                        "Hansen Global Forest Change",
                        "Forest Loss Pixel Share",
                        "Percentage of valid pixels with recorded loss",
                        loss_pct,
                        "%",
                        total,
                        "Share of valid pixels with loss_year > 0. This is a descriptive pixel statistic, not a normalized risk score."
                    )

                    add_metric(
                        "Hansen Global Forest Change",
                        "Forest Loss Pixel Count",
                        "Count",
                        loss_pixels,
                        "pixels",
                        total,
                        "Count of valid pixels with loss_year > 0."
                    )

                    # Distribution of loss years
                    loss_year_values = values[
                        values > 0
                    ]

                    if len(loss_year_values) > 0:

                        add_metric(
                            "Hansen Global Forest Change",
                            "Forest Loss Year",
                            "Median loss-year code",
                            np.median(loss_year_values),
                            "Hansen loss-year code",
                            len(loss_year_values),
                            "Median among pixels with recorded forest loss."
                        )

                        add_metric(
                            "Hansen Global Forest Change",
                            "Forest Loss Year",
                            "Minimum loss-year code",
                            np.min(loss_year_values),
                            "Hansen loss-year code",
                            len(loss_year_values),
                            "Earliest recorded loss-year code among loss pixels."
                        )

                        add_metric(
                            "Hansen Global Forest Change",
                            "Forest Loss Year",
                            "Maximum loss-year code",
                            np.max(loss_year_values),
                            "Hansen loss-year code",
                            len(loss_year_values),
                            "Latest recorded loss-year code among loss pixels."
                        )

                    print(
                        f"   Loss pixel share: "
                        f"{loss_pct:.4f}%"
                    )

            except Exception as e:

                print(
                    f"   Hansen loss_year calculation failed: {e}"
                )


        # ---------------------------------------------------------------------
        # FOREST GAIN
        # ---------------------------------------------------------------------

        if "forest_gain" in ds:

            try:

                da = clip_to_paglam(
                    ds["forest_gain"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                if len(values) > 0:

                    gain_pixels = np.sum(
                        values > 0
                    )

                    total = len(values)

                    gain_pct = (
                        gain_pixels / total * 100
                        if total > 0
                        else np.nan
                    )

                    add_metric(
                        "Hansen Global Forest Change",
                        "Forest Gain Pixel Share",
                        "Percentage of valid pixels with recorded gain",
                        gain_pct,
                        "%",
                        total,
                        "Share of valid pixels with forest gain."
                    )

                    print(
                        f"   Gain pixel share: "
                        f"{gain_pct:.4f}%"
                    )

            except Exception as e:

                print(
                    f"   Hansen forest_gain calculation failed: {e}"
                )


# =============================================================================
# 18. FOREST LANDSCAPE INTEGRITY INDEX
# =============================================================================

if "Forest Landscape Integrity Index" in loaded_data:

    print("\nForest Landscape Integrity Index")

    ds = loaded_data[
        "Forest Landscape Integrity Index"
    ]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )

        if "integrity_score" in ds:

            try:

                da = clip_to_paglam(
                    ds["integrity_score"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                # Explicitly remove common FLII NoData value
                values = values[
                    values != -9999
                ]

                if len(values) > 0:

                    add_metric(
                        "Forest Landscape Integrity Index",
                        "Forest Landscape Integrity",
                        "Mean",
                        np.mean(values),
                        "FLII native score (0–10)",
                        len(values),
                        "Native FLII score. No additional normalization performed."
                    )

                    add_metric(
                        "Forest Landscape Integrity Index",
                        "Forest Landscape Integrity",
                        "Median",
                        np.median(values),
                        "FLII native score (0–10)",
                        len(values),
                        "Median native FLII score."
                    )

                    add_metric(
                        "Forest Landscape Integrity Index",
                        "Forest Landscape Integrity",
                        "Minimum",
                        np.min(values),
                        "FLII native score (0–10)",
                        len(values),
                        "Minimum valid native FLII score."
                    )

                    add_metric(
                        "Forest Landscape Integrity Index",
                        "Forest Landscape Integrity",
                        "Maximum",
                        np.max(values),
                        "FLII native score (0–10)",
                        len(values),
                        "Maximum valid native FLII score."
                    )

                    add_metric(
                        "Forest Landscape Integrity Index",
                        "Forest Landscape Integrity",
                        "Standard Deviation",
                        np.std(values),
                        "FLII native score (0–10)",
                        len(values),
                        "Standard deviation of valid FLII scores."
                    )

                    print(
                        f"   FLII mean: "
                        f"{np.mean(values):.4f}/10"
                    )

            except Exception as e:

                print(
                    f"   FLII calculation failed: {e}"
                )


# =============================================================================
# 19. TROPICAL TREE COVER 10m / 70m
# =============================================================================

for dataset_name in [
    "Tropical Tree Cover 10 m",
    "Tropical Tree Cover 70 m"
]:

    if dataset_name not in loaded_data:
        continue

    print(f"\n{dataset_name}")

    ds = loaded_data[dataset_name]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )

        for variable in ds.data_vars:

            try:

                da = clip_to_paglam(
                    ds[variable]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                if len(values) == 0:
                    continue

                # IMPORTANT:
                # We do NOT assume that 0–1 means fraction or
                # 0–100 means percentage.
                #
                # We report the actual Cecil values and statistics.
                add_metric(
                    dataset_name,
                    variable,
                    "Mean",
                    np.mean(values),
                    "dataset native unit",
                    len(values),
                    "Native dataset value; no unit conversion or 0–10 scoring."
                )

                add_metric(
                    dataset_name,
                    variable,
                    "Median",
                    np.median(values),
                    "dataset native unit",
                    len(values),
                    "Native dataset value; no unit conversion."
                )

                add_metric(
                    dataset_name,
                    variable,
                    "Minimum",
                    np.min(values),
                    "dataset native unit",
                    len(values),
                    "Minimum valid native value."
                )

                add_metric(
                    dataset_name,
                    variable,
                    "Maximum",
                    np.max(values),
                    "dataset native unit",
                    len(values),
                    "Maximum valid native value."
                )

                add_metric(
                    dataset_name,
                    variable,
                    "Standard Deviation",
                    np.std(values),
                    "dataset native unit",
                    len(values),
                    "Standard deviation of native values."
                )

            except Exception as e:

                print(
                    f"   {variable}: calculation failed: {e}"
                )


# =============================================================================
# 20. AQUEDUCT BASELINE ANNUAL
# =============================================================================

def process_aqueduct_dataframe(
    dataset_name,
    df
):

    print(f"\n{dataset_name}")

    if not isinstance(df, pd.DataFrame):

        print(
            "   Object is not a pandas DataFrame."
        )

        return


    print(
        f"   Rows: {len(df)}"
    )

    print(
        f"   Columns: {len(df.columns)}"
    )


    # -------------------------------------------------------------------------
    # Identify numeric columns
    # -------------------------------------------------------------------------

    numeric_columns = df.select_dtypes(
        include=[np.number]
    ).columns.tolist()


    # Exclude obvious geometry / IDs if present
    excluded_keywords = [
        "latitude",
        "longitude",
        "lat",
        "lon",
        "id",
        "area",
        "shape"
    ]


    for col in numeric_columns:

        col_lower = str(col).lower()

        if any(
            key == col_lower
            or col_lower.endswith("_" + key)
            for key in excluded_keywords
        ):
            continue

        try:

            values = pd.to_numeric(
                df[col],
                errors="coerce"
            ).dropna()

            if len(values) == 0:
                continue

            # Remove common NoData values only.
            values = values[
                ~values.isin(
                    NODATA_VALUES
                )
            ]

            if len(values) == 0:
                continue

            # -----------------------------------------------------------------
            # CRITICAL:
            # Preserve Aqueduct's native value.
            # Do NOT transform it into another score.
            # -----------------------------------------------------------------

            add_metric(
                dataset_name,
                str(col),
                "Source value",
                float(values.iloc[0]),
                "Aqueduct native value",
                None,
                "Aqueduct value retained exactly as supplied by the dataset. No artificial 0–10 conversion."
            )

        except Exception as e:

            print(
                f"   {col}: skipped ({e})"
            )


if "Aqueduct Water Risk - Baseline Annual" in loaded_data:

    process_aqueduct_dataframe(
        "Aqueduct Water Risk - Baseline Annual",
        loaded_data[
            "Aqueduct Water Risk - Baseline Annual"
        ]
    )


# =============================================================================
# 21. AQUEDUCT BASELINE MONTHLY
# =============================================================================

if "Aqueduct Water Risk - Baseline Monthly" in loaded_data:

    process_aqueduct_dataframe(
        "Aqueduct Water Risk - Baseline Monthly",
        loaded_data[
            "Aqueduct Water Risk - Baseline Monthly"
        ]
    )


# =============================================================================
# 22. AQUEDUCT FUTURE ANNUAL
# =============================================================================

if "Aqueduct Water Risk - Future Annual" in loaded_data:

    process_aqueduct_dataframe(
        "Aqueduct Water Risk - Future Annual",
        loaded_data[
            "Aqueduct Water Risk - Future Annual"
        ]
    )


# =============================================================================
# 23. POPULATION
# =============================================================================

if "Population Counts 1 km" in loaded_data:

    print("\nPopulation Counts 1 km")

    ds = loaded_data[
        "Population Counts 1 km"
    ]

    if isinstance(ds, xr.Dataset):

        print(
            "Variables:",
            list(ds.data_vars)
        )

        if "population_count" in ds:

            try:

                da = clip_to_paglam(
                    ds["population_count"]
                )

                values = clean_numeric(
                    da.values.flatten(),
                    remove_255=True
                )

                if len(values) > 0:

                    add_metric(
                        "Population Counts 1 km",
                        "Population Count",
                        "Mean",
                        np.mean(values),
                        "persons/pixel",
                        len(values),
                        "Contextual population statistic; not converted to a biodiversity score."
                    )

                    add_metric(
                        "Population Counts 1 km",
                        "Population Count",
                        "Median",
                        np.median(values),
                        "persons/pixel",
                        len(values),
                        "Median population count per valid pixel."
                    )

                    add_metric(
                        "Population Counts 1 km",
                        "Population Count",
                        "Minimum",
                        np.min(values),
                        "persons/pixel",
                        len(values),
                        "Minimum valid population count."
                    )

                    add_metric(
                        "Population Counts 1 km",
                        "Population Count",
                        "Maximum",
                        np.max(values),
                        "persons/pixel",
                        len(values),
                        "Maximum valid population count."
                    )

                    add_metric(
                        "Population Counts 1 km",
                        "Population Count",
                        "Standard Deviation",
                        np.std(values),
                        "persons/pixel",
                        len(values),
                        "Standard deviation of population count."
                    )

                    print(
                        f"   Mean population: "
                        f"{np.mean(values):.4f}"
                    )

            except Exception as e:

                print(
                    f"   Population calculation failed: {e}"
                )


# =============================================================================
# 24. ADMINISTRATIVE ZONES
# =============================================================================
#
# These are contextual spatial datasets.
# They are NOT biodiversity scores.
#
# We therefore report the available administrative names rather than
# inventing numerical biodiversity statistics.
# =============================================================================

for dataset_name in [
    "Administrative Zones - ADM0",
    "Administrative Zones - ADM1",
    "Administrative Zones - ADM2"
]:

    if dataset_name not in loaded_data:
        continue

    print(f"\n{dataset_name}")

    df = loaded_data[dataset_name]

    if isinstance(df, pd.DataFrame):

        # Print useful name fields
        name_columns = [
            c for c in df.columns
            if any(
                key in str(c).lower()
                for key in [
                    "shape_name",
                    "name"
                ]
            )
        ]

        for col in name_columns:

            vals = (
                df[col]
                .dropna()
                .astype(str)
                .unique()
            )

            for val in vals:

                add_metric(
                    dataset_name,
                    str(col),
                    "Administrative context",
                    val,
                    "name",
                    None,
                    "Administrative reference only; not a biodiversity score."
                )


# =============================================================================
# 25. CREATE FINAL DATAFRAME
# =============================================================================

paglam_metrics_df = pd.DataFrame(
    metric_rows
)


# =============================================================================
# 26. SORT
# =============================================================================

if not paglam_metrics_df.empty:

    paglam_metrics_df = paglam_metrics_df[
        [
            "site",
            "aoi_id",
            "dataset",
            "metric",
            "statistic",
            "value",
            "unit",
            "valid_pixel_count",
            "notes"
        ]
    ]


# =============================================================================
# 27. DISPLAY FINAL TABLE
# =============================================================================

print("\n")
print("=" * 100)
print("FINAL PAGLAM METRIC TABLE — NO ARTIFICIAL SCORING")
print("=" * 100)

display(
    paglam_metrics_df
)


# =============================================================================
# 28. DATASET SUMMARY
# =============================================================================

print("\n")
print("=" * 100)
print("DATASET SUMMARY")
print("=" * 100)

summary_df = (
    paglam_metrics_df
    .groupby("dataset")
    .agg(
        metric_rows=("metric", "count")
    )
    .reset_index()
)

display(summary_df)


# =============================================================================
# 29. FAILED DATASETS
# =============================================================================

if failed_data:

    print("\n")
    print("=" * 100)
    print("DATASETS THAT COULD NOT BE LOADED")
    print("=" * 100)

    failed_df = pd.DataFrame(
        [
            {
                "dataset": k,
                "reason": v
            }
            for k, v in failed_data.items()
        ]
    )

    display(failed_df)


# =============================================================================
# 30. SPECIFIC NOTE FOR TROPICAL TREE COVER
# =============================================================================

print("\n")
print("=" * 100)
print("TROPICAL TREE COVER STATUS")
print("=" * 100)

for dataset_name in [
    "Tropical Tree Cover 10 m",
    "Tropical Tree Cover 70 m"
]:

    if dataset_name in failed_data:

        print(
            f"{dataset_name}: NOT AVAILABLE FOR THIS AOI"
        )

        print(
            f"Reason: {failed_data[dataset_name]}"
        )

    elif dataset_name in loaded_data:

        print(
            f"{dataset_name}: Loaded and processed."
        )

    else:

        print(
            f"{dataset_name}: No result."
        )


# =============================================================================
# 31. SAVE CSV
# =============================================================================

OUTPUT_FILE = (
    "Paglam_Cecil_Nature_Metrics_Statistics_Only.csv"
)

paglam_metrics_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# =============================================================================
# 32. FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    f"Total metric rows: {len(paglam_metrics_df)}"
)

print(
    f"Saved: {OUTPUT_FILE}"
)

print()
print("IMPORTANT:")
print("✓ No artificial 0–10 normalization")
print("✓ No invented ecological ranking")
print("✓ Categorical rasters reported as class proportions")
print("✓ NoData values excluded where applicable")
print("✓ FLII retained on its native 0–10 scale")
print("✓ Aqueduct values retained on their native Cecil/Aqueduct scale")
print("✓ Population retained as contextual statistics")
print("✓ Age & Sex dataset excluded")
print("✓ Tropical Tree Cover coverage errors are reported, not forced")
print("=" * 100)

PAGLAM CECIL NATURE METRICS — STATISTICS ONLY
AOI_ID: 4db972f3-1895-415e-8b2a-30d2163649da
Age & Sex dataset: EXCLUDED

LOADING / REUSING CECIL DATASETS

SBTN Natural Lands
   Reusing subscription: 56944dec-ba02-43d7-acb6-83b49102182c
   ✓ Raster loaded: ['binary_class', 'class_value']

Land Cover 9-Class
   Reusing subscription: b2f1a3d9-45b8-40fb-b953-518aef07f930
   ✓ Raster loaded: ['land_cover_class']

Global Forest Cover
   Reusing subscription: 4bba3552-4853-4b50-83ed-7fa4b45e8ca0
   ✓ Raster loaded: ['forest_cover']

Global Forest Type
   Reusing subscription: eba6d3b3-d151-461b-b2a5-3484c582ddd3
   ✓ Raster loaded: ['forest_type']

Hansen Global Forest Change
   Reusing subscription: ded8d3bb-9fe0-4a89-8126-6a515b556535
   ✓ Raster loaded: ['data_mask', 'forest_gain', 'loss_year', 'tree_cover']

Forest Landscape Integrity Index
   Reusing subscription: 6b196667-8589-4ac3-994a-82dab15039a7
   ✓ Raster loaded: ['integrity_score']

Tropical Tree Cover 10 m
   Creating subscriptio

,site,aoi_id,dataset,metric,statistic,value,unit,valid_pixel_count,notes
0,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,SBTN Natural Lands,Natural Land Extent,Percentage of valid pixels classified as natural,49.144254,%,3681.0,SBTN binary_class: natural vs non-natural.
1,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,SBTN Natural Lands,Natural Land Pixel Count,Count,1809,pixels,3681.0,Count of valid pixels classified as natural.
2,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,Global Forest Cover,Forest Cover,Percentage of valid pixels classified as forest,9.055592,%,33206.0,Binary forest/non-forest classification. NoDat...
3,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,Global Forest Cover,Forest Pixel Count,Count,3007,pixels,33206.0,Count of valid forest pixels.
4,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,Global Forest Type,Forest Type Class 0,Class proportion,90.944408,%,33206.0,Categorical forest-type raster. Class proporti...
...,...,...,...,...,...,...,...,...,...
428,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,Population Counts 1 km,Population Count,Maximum,58.325943,persons/pixel,32.0,Maximum valid population count.
429,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,Population Counts 1 km,Population Count,Standard Deviation,22.046607,persons/pixel,32.0,Standard deviation of population count.
430,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,Administrative Zones - ADM0,shape_name,Administrative context,India,name,NaN,Administrative reference only; not a biodivers...
431,Paglam_CR_Demarcation2025,4db972f3-1895-415e-8b2a-30d2163649da,Administrative Zones - ADM1,shape_name,Administrative context,Assam,name,NaN,Administrative reference only; not a biodivers...




DATASET SUMMARY


,dataset,metric_rows
0,Administrative Zones - ADM0,1
1,Administrative Zones - ADM1,1
2,Administrative Zones - ADM2,1
3,Aqueduct Water Risk - Baseline Annual,169
4,Aqueduct Water Risk - Baseline Monthly,108
5,Aqueduct Water Risk - Future Annual,126
6,Forest Landscape Integrity Index,5
7,Global Forest Cover,2
8,Global Forest Type,2
9,Hansen Global Forest Change,11




DATASETS THAT COULD NOT BE LOADED


,dataset,reason
0,Tropical Tree Cover 10 m,"Unprocessable Content (422)\n[\n ""AOI maximum..."
1,Tropical Tree Cover 70 m,"Unprocessable Content (422)\n[\n ""AOI maximum..."




TROPICAL TREE COVER STATUS
Tropical Tree Cover 10 m: NOT AVAILABLE FOR THIS AOI
Reason: Unprocessable Content (422)
[
  "AOI maximum latitude: 23.44 degrees"
]
Tropical Tree Cover 70 m: NOT AVAILABLE FOR THIS AOI
Reason: Unprocessable Content (422)
[
  "AOI maximum latitude: 23.44 degrees"
]


DONE
Total metric rows: 433
Saved: Paglam_Cecil_Nature_Metrics_Statistics_Only.csv

IMPORTANT:
✓ No artificial 0–10 normalization
✓ No invented ecological ranking
✓ Categorical rasters reported as class proportions
✓ NoData values excluded where applicable
✓ FLII retained on its native 0–10 scale
✓ Aqueduct values retained on their native Cecil/Aqueduct scale
✓ Population retained as contextual statistics
✓ Age & Sex dataset excluded
✓ Tropical Tree Cover coverage errors are reported, not forced


In [54]:
from google.colab import sheets
import numpy as np
import pandas as pd

# Create a copy and convert all columns to object type to make element-level mapping robust
sheets_df = paglam_metrics_df.copy().astype(object)

# Apply custom map to convert any leftover numpy types or pandas nullable structures to python native types
def sanitize_value(val):
    if isinstance(val, (np.int64, np.integer)):
        return int(val)
    elif isinstance(val, (np.float64, np.floating)):
        return float(val) if not np.isnan(val) else None
    elif pd.isna(val):
        return None
    return val

sheets_df = sheets_df.map(sanitize_value)

sheet = sheets.InteractiveSheet(df=sheets_df)

https://docs.google.com/spreadsheets/d/1zlbAdjctH_Y14wI-YHiFoX8lCLYwWJEk4jSxBloTv7A/edit#gid=0
